## BirdCLEF 2026 - data download

The cell below downloads the competition data from Kaggle, but only if it is not already in the `data/` folder. Complete these steps once before running it.

**1. Verify your Kaggle account**
1. Sign in (or create an account) at [kaggle.com](https://www.kaggle.com).
2. Go to **Settings → Phone verification, Identity verification** and verify yourself. Kaggle blocks competition data downloads for unverified accounts.

**2. Accept the competition rules**
1. Open the [BirdCLEF 2026 competition](https://www.kaggle.com/competitions/birdclef-2026).
2. Click **Join Competition** / **I Understand and Accept**.

**3. Create a Kaggle API token**
1. Go to **Settings → API** on kaggle.com and create a new token (it starts with `KGAT_`).
2. Copy it right away - Kaggle shows it only once. Never share it or commit it to git.

**4. Make the token available to the notebook**
The token must be stored under the name `KAGGLE_API_TOKEN`. Use one of these options:

**Option A - Colab secrets (when running in Google Colab)**
1. Click the key icon (**Secrets**) in the left sidebar.
2. Add a new secret named `KAGGLE_API_TOKEN` with your token as the value.
3. Switch on **Notebook access** for it.

**Option B - `.env` file (local, VS Code, or Colab)**
1. Copy `.env.example` to `.env` and set `KAGGLE_API_TOKEN=your_token_here`.
2. Running in Colab: upload the `.env` file to the Colab working directory (**Files** panel → upload, usually `/content`). It is lost when the session ends, so upload it again each session.

**5. Run the cell**
Run the next cell. It downloads and extracts the data (~16 GB, so make sure there is enough free disk space). If the data is already there, it only prints the path.

In [15]:
import os
import subprocess
import sys
from pathlib import Path

COMPETITION = "birdclef-2026"
KAGGLE_TOKEN = "KAGGLE_API_TOKEN"  # name of the Colab secret / .env variable
DATA_ROOT = Path.cwd() / "data"  # git-ignored; kagglehub stores its cache here

# Files and folders that must exist for the data to count as complete
REQUIRED = [
    "train.csv",
    "taxonomy.csv",
    "train_soundscapes_labels.csv",
    "train_audio",
    "train_soundscapes",
]


def is_complete(folder):
    return all((folder / name).exists() for name in REQUIRED)


def find_local_data():
    """Return the folder holding the competition files, or None."""
    for train_csv in DATA_ROOT.rglob("train.csv"):
        if is_complete(train_csv.parent):
            return train_csv.parent
    return None


def load_from_colab_secrets():
    """Read the token from Colab Secrets; return False outside Colab."""
    try:
        from google.colab import userdata
    except ImportError:
        return False

    try:
        value = userdata.get(KAGGLE_TOKEN)
    except Exception:  # secret not defined or notebook access off
        return True
    if value:
        os.environ[KAGGLE_TOKEN] = value.strip()
    return True


def load_from_env_file():
    """Read the token from the nearest .env file (current folder or a parent)."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        env_file = folder / ".env"
        if not env_file.is_file():
            continue
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip().removeprefix("export ").strip()
            value = value.strip().strip("\"'")
            if key == KAGGLE_TOKEN and value:
                os.environ[KAGGLE_TOKEN] = value
        print(f"Loaded .env from {env_file}")
        return True
    return False


DATA_DIR = find_local_data()

if DATA_DIR:
    print(f"Data already available: {DATA_DIR}")
else:
    print(f"Data not found in {DATA_ROOT}, downloading from Kaggle...")

    # Token: 1. Colab secrets, 2. .env file
    load_from_colab_secrets()
    if not os.environ.get(KAGGLE_TOKEN):
        load_from_env_file()
    if not os.environ.get(KAGGLE_TOKEN):
        raise EnvironmentError(
            f"{KAGGLE_TOKEN} not found. Add it as a Colab secret (notebook "
            f"access on) or to a .env file. Working directory: {Path.cwd()}"
        )

    # Keep the kagglehub cache inside the repo (must be set before the import).
    os.environ["KAGGLEHUB_CACHE"] = str(DATA_ROOT)

    try:
        import kagglehub
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "kagglehub"],
            check=True,
        )
        import kagglehub

    # Authenticate explicitly with the token
    from kagglehub.config import set_kaggle_api_token

    set_kaggle_api_token(os.environ[KAGGLE_TOKEN])

    # Fails here with a clear error if the token is rejected.
    print("Authenticated as:", kagglehub.whoami()["username"])

    # The competition rules must be accepted on kaggle.com (same account).
    DATA_DIR = Path(kagglehub.competition_download(COMPETITION))
    print("Path to competition files:", DATA_DIR)

    if not is_complete(DATA_DIR):
        missing = [n for n in REQUIRED if not (DATA_DIR / n).exists()]
        raise FileNotFoundError(f"Download finished but missing: {missing}")

Data already available: m:\Bird-CLEF\BridCLEF-team-buzby\data\competitions\birdclef-2026


Read metadata of audio files

In [16]:
import pandas as pd
from tinytag import TinyTag

train_audio_metadata_csv = DATA_DIR / "train_audio_metadata.csv"
train_soundscapes_metadata_csv = DATA_DIR / "train_soundscapes_metadata.csv"

# List train audio files
train_audio_dir = DATA_DIR / "train_audio"
train_audio_files = sorted(p.relative_to(train_audio_dir).as_posix() for p in train_audio_dir.glob("*/*.ogg"))

# List soundscape files
train_soundscapes_dir = DATA_DIR / "train_soundscapes"
train_soundscapes_files = sorted(p.relative_to(train_soundscapes_dir).as_posix() for p in train_soundscapes_dir.glob("*.ogg"))

def get_audio_metadata(audio_path: list[str|Path]|str|Path, csv_output: str|Path|None=None) -> pd.DataFrame:
    if isinstance(audio_path, (str, Path)):
        audio_path = [audio_path]
    metadata_list = []
    for path in audio_path:
        audio = TinyTag.get(path)
        metadata_list.append({
            "filename": path,
            "length": audio.duration,
            "channels": audio.channels,
            "sample_rate": audio.samplerate,
            "bitrate": audio.bitrate
        })
    metadata_df = pd.DataFrame(metadata_list)
    if csv_output:
        metadata_df.to_csv(csv_output, index=False)
    return metadata_df

if train_audio_metadata_csv.exists():
    train_audio_metadata_df = pd.read_csv(train_audio_metadata_csv)
else:
    train_audio_metadata_df = get_audio_metadata([train_audio_dir / f for f in train_audio_files], train_audio_metadata_csv)
train_audio_metadata_df.set_index("filename", inplace=True)

if train_soundscapes_metadata_csv.exists():
    train_soundscapes_metadata_df = pd.read_csv(train_soundscapes_metadata_csv)
else:
    train_soundscapes_metadata_df = get_audio_metadata([train_soundscapes_dir / f for f in train_soundscapes_files], train_soundscapes_metadata_csv)
train_soundscapes_metadata_df.set_index("filename", inplace=True)

print(train_audio_metadata_df.head())
print(train_soundscapes_metadata_df.head())

                                                       length  channels  \
filename                                                                  
m:\Bird-CLEF\BridCLEF-team-buzby\data\competiti...  28.392000         1   
m:\Bird-CLEF\BridCLEF-team-buzby\data\competiti...  18.024000         1   
m:\Bird-CLEF\BridCLEF-team-buzby\data\competiti...   0.288000         1   
m:\Bird-CLEF\BridCLEF-team-buzby\data\competiti...  15.782875         1   
m:\Bird-CLEF\BridCLEF-team-buzby\data\competiti...  71.064000         1   

                                                    sample_rate  bitrate  
filename                                                                  
m:\Bird-CLEF\BridCLEF-team-buzby\data\competiti...        32000     86.0  
m:\Bird-CLEF\BridCLEF-team-buzby\data\competiti...        32000     86.0  
m:\Bird-CLEF\BridCLEF-team-buzby\data\competiti...        32000     86.0  
m:\Bird-CLEF\BridCLEF-team-buzby\data\competiti...        32000     86.0  
m:\Bird-CLEF\BridCLEF-te

In [17]:
train_audio_unique_sample_rates = train_audio_metadata_df["sample_rate"].unique()
print(f"Train audio sample rates: {len(train_audio_unique_sample_rates)} unique values")
print(train_audio_unique_sample_rates)

train_soundscapes_unique_sample_rates = train_soundscapes_metadata_df["sample_rate"].unique()
print(f"Train soundscapes sample rates: {len(train_soundscapes_unique_sample_rates)} unique values")
print(train_soundscapes_unique_sample_rates)

train_audio_unique_durations = train_audio_metadata_df["length"].unique()
print(f"Train audio durations: {len(train_audio_unique_durations)} unique values")
print(train_audio_unique_durations)

train_soundscapes_unique_durations = train_soundscapes_metadata_df["length"].unique()
print(f"Train soundscapes durations: {len(train_soundscapes_unique_durations)} unique values")
print(train_soundscapes_unique_durations)

Train audio sample rates: 1 unique values
[32000]
Train soundscapes sample rates: 1 unique values
[32000]
Train audio durations: 16596 unique values
[28.392      18.024       0.288      ...  4.43240625 13.0694375
 16.96      ]
Train soundscapes durations: 1 unique values
[60.]


In [18]:
# Taxonomy
taxonomy_df = pd.read_csv(DATA_DIR / "taxonomy.csv")
print("\nTaxonomy:")
print(taxonomy_df.head())
print(f"Taxonomy has {len(taxonomy_df)} rows and {taxonomy_df['primary_label'].nunique()} unique primary labels.")
assert taxonomy_df['primary_label'].is_unique, "Primary labels in taxonomy.csv are not unique."
taxonomy_df.set_index('primary_label', inplace=True)
print(f"Taxonomy has {taxonomy_df['scientific_name'].nunique()} unique animal scientific names.")
assert taxonomy_df['scientific_name'].is_unique, "Scientific names in taxonomy.csv are not unique."
print(f"The number of animals in the classes of taxonomy: ")
print(taxonomy_df['class_name'].value_counts())


Taxonomy:
  primary_label  inat_taxon_id         scientific_name  \
0       1161364        1161364            Guyalna cuta   
1        116570         116570           Caiman yacare   
2       1176823        1176823  Leptodactylus luctator   
3       1491113        1491113       Adenomera guarani   
4       1595929        1595929       Lysapsus limellum   

                  common_name class_name  
0                Guyalna cuta    Insecta  
1  Southern Spectacled Caiman   Reptilia  
2               Wrestler Frog   Amphibia  
3    Guaraní leaf-litter frog   Amphibia  
4      Uruguay Harlequin Frog   Amphibia  
Taxonomy has 234 rows and 234 unique primary labels.
Taxonomy has 234 unique animal scientific names.
The number of animals in the classes of taxonomy: 
class_name
Aves        162
Amphibia     35
Insecta      28
Mammalia      8
Reptilia      1
Name: count, dtype: int64


In [19]:
# Read train CSV
train_audio_label_df = pd.read_csv(DATA_DIR / "train.csv")
print("Train audio labels:")
print(train_audio_label_df.head())
# Check if filanames are unique in the table
assert train_audio_label_df['filename'].is_unique, "Filenames in train.csv are not unique."
train_audio_label_df.set_index("filename", inplace=True)

# Number of species and recordings per class, using the primary label and the class from taxonomy



print("\nNumber of species per class in train.csv:")



Train audio labels:
  primary_label secondary_labels type  latitude  longitude scientific_name  \
0       1161364               []   []  -22.7562   -46.8666    Guyalna cuta   
1       1161364               []   []  -22.7558   -46.8700    Guyalna cuta   
2       1161364               []   []  -22.7547   -46.8728    Guyalna cuta   
3       1161364               []   []  -22.7547   -46.8728    Guyalna cuta   
4       1161364               []   []  -22.7426   -46.8985    Guyalna cuta   

    common_name class_name  inat_taxon_id         author   license  rating  \
0  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
1  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
2  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
3  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
4  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   

                                          

In [20]:
# Soundscape labels
soundscapes_df = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")
print("\nSoundscape labels:")
print(soundscapes_df.head())


Soundscape labels:
                                    filename     start       end  \
0  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:00  00:00:05   
1  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:05  00:00:10   
2  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:10  00:00:15   
3  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:15  00:00:20   
4  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:20  00:00:25   

                    primary_label  
0  22961;23158;24321;517063;65380  
1  22961;23158;24321;517063;65380  
2  22961;23158;24321;517063;65380  
3  22961;23158;24321;517063;65380  
4  22961;23158;24321;517063;65380  
